# DocRestore — Model Comparison: DocRes vs NAFNet

**Owner:** Sakshat  
**Week 3**

Loads evaluation results produced by Apoorva's `eval/run_eval.py` and compares
DocRes and NAFNet on PSNR, SSIM, and OCR character error rate (CER).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

# Add project root so we can import demo/inference if needed
PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR  = PROJECT_ROOT / "eval" / "outputs"
DOCRES_CSV   = RESULTS_DIR / "results_docres.csv"
NAFNET_CSV   = RESULTS_DIR / "results_nafnet.csv"

print("DocRes results :", DOCRES_CSV)
print("NAFNet results :", NAFNET_CSV)

## 1. Load Results

In [ ]:
df_docres = pd.read_csv(DOCRES_CSV)
df_nafnet = pd.read_csv(NAFNET_CSV)

print("DocRes columns :", df_docres.columns.tolist())
print("NAFNet columns :", df_nafnet.columns.tolist())
print(f"\nDocRes rows : {len(df_docres)}")
print(f"NAFNet rows : {len(df_nafnet)}")

df_docres.head()

## 2. Summary Statistics

In [ ]:
metrics = ["psnr", "ssim", "cer"]

summary = pd.DataFrame({
    "DocRes": df_docres[metrics].mean(),
    "NAFNet": df_nafnet[metrics].mean(),
}).T

summary["psnr"] = summary["psnr"].map("{:.2f} dB".format)
summary["ssim"] = summary["ssim"].map("{:.4f}".format)
summary["cer"]  = summary["cer"].map("{:.2%}".format)

print(summary.to_string())

## 3. Bar Charts

In [ ]:
# Numeric means for plotting
means = {
    "PSNR (dB)": [df_docres["psnr"].mean(), df_nafnet["psnr"].mean()],
    "SSIM":      [df_docres["ssim"].mean(), df_nafnet["ssim"].mean()],
    "CER (%)": [df_docres["cer"].mean() * 100, df_nafnet["cer"].mean() * 100],
}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = ["#4C72B0", "#DD8452"]
x = np.arange(2)
labels = ["DocRes", "NAFNet"]

for ax, (metric_name, values) in zip(axes, means.items()):
    bars = ax.bar(x, values, color=colors, width=0.5)
    ax.set_title(metric_name, fontsize=13)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.bar_label(bars, fmt="%.3f", padding=3)
    if "CER" in metric_name:
        ax.set_ylabel("Lower is better")
    else:
        ax.set_ylabel("Higher is better")
    ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("DocRes vs NAFNet — Mean Metrics on Test Set", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "model_comparison_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → eval/outputs/model_comparison_bar.png")

## 4. Side-by-Side Visual Examples (5 test images)

In [ ]:
# The results CSVs are expected to have columns:
#   image_path, psnr, ssim, cer, restored_path (optional)
# If restored_path is present we load it; otherwise we show the degraded input.

N = 5
sample_docres = df_docres.head(N)
sample_nafnet = df_nafnet.head(N)

def _load(path_str):
    p = Path(path_str)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    return Image.open(p).convert("RGB")

fig, axes = plt.subplots(N, 3, figsize=(12, 4 * N))
col_titles = ["Degraded (input)", "DocRes output", "NAFNet output"]

for i in range(N):
    row_d = sample_docres.iloc[i]
    row_n = sample_nafnet.iloc[i]

    degraded_img = _load(row_d["image_path"])
    docres_img   = _load(row_d["restored_path"]) if "restored_path" in row_d.index else degraded_img
    nafnet_img   = _load(row_n["restored_path"]) if "restored_path" in row_n.index else degraded_img

    for j, (img, title) in enumerate(zip(
        [degraded_img, docres_img, nafnet_img], col_titles
    )):
        axes[i][j].imshow(img)
        axes[i][j].axis("off")
        if i == 0:
            axes[i][j].set_title(title, fontsize=12)
    axes[i][1].set_title(
        f"DocRes  PSNR={row_d['psnr']:.2f} SSIM={row_d['ssim']:.3f}", fontsize=9
    )
    axes[i][2].set_title(
        f"NAFNet  PSNR={row_n['psnr']:.2f} SSIM={row_n['ssim']:.3f}", fontsize=9
    )

plt.suptitle("5 Test Image Outputs: DocRes vs NAFNet", fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "model_comparison_examples.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved → eval/outputs/model_comparison_examples.png")

## 5. Analysis

*Fill in after results are available.*

**Which model wins?**

- **PSNR**: [DocRes / NAFNet] achieves higher PSNR (___dB vs ___dB), indicating [better / similar] pixel-level fidelity.
- **SSIM**: [DocRes / NAFNet] scores higher SSIM (___ vs ___), suggesting [sharper structure / more perceptually similar] outputs.
- **CER**: [DocRes / NAFNet] yields lower CER (___% vs ___%),  which directly translates to more OCR-readable output.

**Why?**

DocRes was pre-trained specifically on document images and fine-tuned here, giving it an advantage on document-specific degradations (ink bleed, fold marks). NAFNet is a general-purpose restoration backbone trained from scratch on our ~900-pair synthetic set, which limits its performance but demonstrates that a lightweight model can still learn meaningful restoration without pretraining.